# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane (kept, not switched):** Refresh / Content Opportunity Scoring. The decision this notebook supports is *"out of thousands of pages, which one should a human look at first?"*

This notebook builds the simple hand-written rule that my Week-5 model has to beat. It uses the 30,000-row starter file, one row per page. Everything is described as observed, measured, directional or decision-support.

## 0. Setup

Loads the starter file and builds the outcome column `declining` (the page's `trend_direction` is `down`). **The outcome is only used to check the signals and to grade the finished rule. It is never used to build the score.**

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    # local run: walk up until we find the repo root (the folder that holds data/raw)
    while not os.path.isfile("data/raw/content_refresh_anonymized.csv") and os.getcwd() != "/":
        os.chdir("..")

import json
from pathlib import Path
import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The outcome. It is used ONLY to grade the signals and the finished rule - never to build the score.
df["declining"] = (df["trend_direction"] == "down").astype(int)
print(f"{len(df):,} pages, {df['client_id'].nunique()} clients | overall declining rate: {df['declining'].mean():.1%}")


30,000 pages, 32 clients | overall declining rate: 54.2%


## 1. My rule and its reason codes

**The rule in plain words.** A page is worth reviewing first when Google already shows it a lot, but searchers click it much less than they usually click a page at that position. The further its click-through rate (CTR) sits below the typical CTR for its position, and the more impressions it has, the higher it ranks. If the page has also not been updated for more than 90 days, the suggested action is to refresh the content as well as rewrite the title.

**The score.** `score = ctr_gap x log(1 + impressions_90d)`, where `ctr_gap` runs from 0 (at or above the typical CTR for its position band) to 1 (zero CTR). There are no fitted weights, so anyone can recompute a score by hand.

**Reason codes (one per page) and the action each one gives:**

- `low_ctr_for_position` -> *Rewrite title/meta description*
- `low_ctr_stale_page` -> *Refresh the content and rewrite title/meta* (same CTR problem, and not updated for more than 90 days)
- `not_flagged` -> *No action* (CTR is at or above typical; score 0, left out of the queue)

**Which pages the rule looks at.** Only pages with a real average position of 20 or better (pages 1-2 of Google) and at least 300 impressions in 90 days. Below position 20, a CTR near zero is normal, so it says nothing. `avg_position = 0` means "no data", so those rows are excluded.

**Before I trust the rule I check the two signals it leans on.** Both sit behind real FlyRank flag ideas: *CTR versus position* (behind the CTR-fix logic) and *staleness* (behind the refresh flags). Each gets a bucket table with n printed and a one-word verdict.

In [2]:
# The slice my rule works on (two plain filters, no outcome used):
MIN_IMPRESSIONS = 300     # enough exposure for a CTR to mean something
MAX_POSITION    = 20      # pages 1-2 of Google; below that CTR is naturally ~0
STALE_DAYS      = 90      # "not updated for more than 90 days"

has_pos = df["avg_position"] > 0                 # avg_position == 0 means "no data", not rank zero
elig = df[has_pos & (df["avg_position"] <= MAX_POSITION) & (df["impressions_90d"] >= MIN_IMPRESSIONS)].copy()

print(f"pages with a real position: {has_pos.sum():,}")
print(f"pages in my slice (position 1-{MAX_POSITION}, >= {MIN_IMPRESSIONS} impressions): {len(elig):,} from {elig['client_id'].nunique()} clients")
print(f"declining rate inside the slice (the base rate to beat): {elig['declining'].mean():.1%}")

# Position bands + the typical CTR in each band (all rows, no labels used).
BAND_EDGES  = [0, 3, 6, 10, 15, 20]
BAND_LABELS = ["1-3", "4-6", "7-10", "11-15", "16-20"]
elig["position_band"] = pd.cut(elig["avg_position"], BAND_EDGES, labels=BAND_LABELS)
elig["band_median_ctr"] = elig.groupby("position_band", observed=True)["ctr"].transform("median")

# CTR gap: how far BELOW the typical CTR for its position band (0 = at or above typical, 1 = zero CTR).
elig["ctr_gap"] = (1 - elig["ctr"] / elig["band_median_ctr"]).clip(0, 1)
elig["stale"] = (elig["days_since_last_update"] > STALE_DAYS).astype(int)

band_tab = (elig.groupby("position_band", observed=True)
            .agg(n=("ctr", "size"), typical_ctr_pct=("band_median_ctr", "first"))
            .round(2))
band_tab


pages with a real position: 28,795
pages in my slice (position 1-20, >= 300 impressions): 13,205 from 29 clients
declining rate inside the slice (the base rate to beat): 60.7%


                  n  typical_ctr_pct
position_band                       
1-3             507             0.20
4-6            3140             0.29
7-10           4507             0.20
11-15          3037             0.18
16-20          2014             0.16

### Signal 1 — CTR below the norm for its position (flag-linked: CTR-fix logic)

**Claim:** pages whose CTR is well below the typical CTR for their position are more likely to be declining.
**Test:** bucket the pages by how far below the typical CTR of their position band they sit, then print the declining rate and n for each bucket. Then repeat inside each position band, and on two different halves of the clients. I only trust a bucket with at least 50 pages.

In [3]:
# Signal 1 (flag-linked, CTR-vs-position): is a page whose CTR is BELOW the norm for its position more likely to be declining?
base = elig["declining"].mean()
elig["gap_bucket"] = pd.cut(elig["ctr_gap"], [-0.01, 0, 0.25, 0.5, 0.75, 1.01],
                            labels=["at/above typical", "0-25% below", "25-50% below", "50-75% below", "75-100% below"])
sig1 = (elig.groupby("gap_bucket", observed=True)["declining"]
        .agg(n="size", declining_rate="mean").round(3))
print(f"base rate in the slice: {base:.3f}")
display(sig1)

# Does it hold inside every position band? (low-CTR half vs high-CTR half of that band)
rows = []
for b, g in elig.groupby("position_band", observed=True):
    low = g[g["ctr"] < g["band_median_ctr"]]
    high = g[g["ctr"] >= g["band_median_ctr"]]
    rows.append((b, len(low), low["declining"].mean(), len(high), high["declining"].mean()))
within = pd.DataFrame(rows, columns=["band", "n_below_typical", "declining_below", "n_at_or_above", "declining_at_or_above"]).round(3)
within["difference"] = (within["declining_below"] - within["declining_at_or_above"]).round(3)
display(within)

# Replication on a different slice: split the clients in two halves and repeat the bucket table on each.
clients = np.array(sorted(elig["client_id"].unique()))
rng = np.random.RandomState(0)
half_a = set(rng.choice(clients, len(clients) // 2, replace=False))
for name, mask in [("client half A", elig["client_id"].isin(half_a)), ("client half B", ~elig["client_id"].isin(half_a))]:
    g = elig[mask]
    top = g[g["ctr_gap"] > 0.75]["declining"]
    bot = g[g["ctr_gap"] == 0]["declining"]
    print(f"{name}: n={len(g):,} | at/above typical {bot.mean():.3f} (n={len(bot):,}) -> 75-100% below {top.mean():.3f} (n={len(top):,})")


base rate in the slice: 0.607


                     n  declining_rate
gap_bucket                            
at/above typical  6718           0.546
0-25% below       1254           0.592
25-50% below      1670           0.612
50-75% below      1267           0.678
75-100% below     2296           0.752

    band  n_below_typical  declining_below  n_at_or_above  \
0    1-3              246            0.882            261   
1    4-6             1549            0.695           1591   
2   7-10             2204            0.639           2303   
3  11-15             1500            0.644           1537   
4  16-20              988            0.690           1026   

   declining_at_or_above  difference  
0                  0.625       0.257  
1                  0.514       0.181  
2                  0.529       0.110  
3                  0.572       0.072  
4                  0.574       0.116  

client half A: n=10,382 | at/above typical 0.528 (n=5,153) -> 75-100% below 0.731 (n=1,771)
client half B: n=2,823 | at/above typical 0.605 (n=1,565) -> 75-100% below 0.823 (n=525)


**Verdict: CONFIRMED (directional).**

The declining rate climbs in every step: 54.6% for pages at or above the typical CTR, then 59.2%, 61.2%, 67.8% and 75.2% for pages 75-100% below it. The smallest bucket still has 1,254 pages. The difference is positive in all five position bands (from +7 points in the 11-15 band to +26 points in the 1-3 band), and the same direction shows up in both halves of the clients (about 53% to 73% in half A, about 60% to 82% in half B).

**What this does not show:** it shows that low-CTR pages are *more often declining* in this file. It does not show that rewriting a title would *fix* a page. It is a measured association on one 90-day snapshot, so I treat it as decision-support for where to look first.

### Signal 2 — staleness (flag-linked: the refresh flags)

**Claim:** pages that have not been updated for a long time are more likely to be declining.
**Test:** declining rate and n by freshness tier, then "not updated for more than 90 days" against "updated within 90 days". Then two skeptic checks, because staleness can be a client effect: (1) compare stale and fresh pages after removing each client's own average, (2) look at what `days_since_last_update` actually contains.

In [4]:
# Signal 2 (flag-linked, staleness behind the refresh flags): are pages not updated for a long time more likely to be declining?
sig2 = (elig.groupby("freshness_tier", observed=True)["declining"]
        .agg(n="size", declining_rate="mean").round(3))
display(sig2)

two = (elig.groupby("stale")["declining"].agg(n="size", declining_rate="mean").round(3))
two.index = ["updated within 90 days", "not updated for > 90 days"]
display(two)

# Skeptic check 1: is "stale" just a client effect? Compare stale vs fresh AFTER removing each client's own average.
elig["client_mean"] = elig.groupby("client_id")["declining"].transform("mean")
elig["above_client_avg"] = elig["declining"] - elig["client_mean"]
within_client = elig.groupby("stale")["above_client_avg"].mean().round(3)
print("average decline rate relative to the page's own client (0 = typical for that client):")
print(f"   updated within 90 days: {within_client[0]:+.3f}   not updated > 90 days: {within_client[1]:+.3f}")

# Skeptic check 2: what does days_since_last_update actually look like?
top_vals = df["days_since_last_update"].value_counts().head(3)
print("\nmost common values of days_since_last_update across all 30,000 pages:")
for v, c in top_vals.items():
    print(f"   {v} days: {c:,} pages ({c / len(df):.1%})")
share_one_value = (df.groupby("client_id")["days_since_last_update"]
                   .agg(lambda s: s.value_counts(normalize=True).iloc[0]))
print(f"clients where ONE value covers more than half of their pages: {(share_one_value > 0.5).sum()} of {len(share_one_value)}")

# Replication on a different slice: the same two client halves as signal 1.
for name, mask in [("client half A", elig["client_id"].isin(half_a)), ("client half B", ~elig["client_id"].isin(half_a))]:
    g = elig[mask]
    s0, s1 = g[g["stale"] == 0]["declining"], g[g["stale"] == 1]["declining"]
    print(f"{name}: updated within 90 days {s0.mean():.3f} (n={len(s0):,}) vs not updated > 90 days {s1.mean():.3f} (n={len(s1):,})")


                   n  declining_rate
freshness_tier                      
0-30            8393           0.595
181+              14           0.857
31-90             80           0.525
91-180          4718           0.628

                              n  declining_rate
updated within 90 days     8473           0.595
not updated for > 90 days  4732           0.629

average decline rate relative to the page's own client (0 = typical for that client):
   updated within 90 days: -0.017   not updated > 90 days: +0.030

most common values of days_since_last_update across all 30,000 pages:
   20 days: 11,573 pages (38.6%)
   104 days: 8,773 pages (29.2%)
   22 days: 3,564 pages (11.9%)
clients where ONE value covers more than half of their pages: 23 of 32


client half A: updated within 90 days 0.583 (n=6,317) vs not updated > 90 days 0.601 (n=4,065)
client half B: updated within 90 days 0.630 (n=2,156) vs not updated > 90 days 0.796 (n=667)


**Verdict: MIXED.**

At first glance staleness looks like a small plus: 62.9% of the pages not updated for more than 90 days are declining, against 59.5% of pages updated within 90 days (n = 4,732 and 8,473). The tiers are not clean, though. The 91-180 tier is 62.8% (n = 4,718), but the 31-90 tier has only 80 pages (52.5%) and the 181+ tier has only 14 (85.7%, below my 50-page floor). I will not read anything into those two.

Two things hold the signal back. After removing each client's own average, stale pages sit +3.0 points above their client's average and fresh pages -1.7 points below it, so the gap is real but small (about 4.7 points). And the column is very lumpy: 38.6% of all pages have exactly 20 days and 29.2% have exactly 104 days, and in 23 of 32 clients a single value covers more than half of the pages. That looks like clients updating pages in batches, so "stale" often means "belongs to a client whose last batch was 104 days ago". The direction repeats in both client halves (58.3% vs 60.1% in half A, 63.0% vs 79.6% in half B), but half A shows almost nothing.

**What it means for the rule:** staleness is real enough to change the *action* (refresh and rewrite, instead of rewrite only) but too weak to change the *rank*. That is why it is not in the score.

## 2. Build the ranked queue (writes the CSV)

The cell below codes the score, attaches one reason code and one action label to each page, ranks the pages, and writes the flagged ones to `work/outputs/baseline_action_score.csv` (the file stays out of git on purpose, the notebook regenerates it on every run). Ties are broken by impressions and then `content_id`, so the order is the same on every run. It also writes a small metrics file, `work/outputs/w04_baseline_metrics.json`.

In [5]:
# The rule, in code. Readable on purpose: no fitted weights, no model.
elig["score"] = elig["ctr_gap"] * np.log1p(elig["impressions_90d"])

elig["reason_code"] = np.where(elig["ctr_gap"] == 0, "not_flagged",
                        np.where(elig["stale"] == 1, "low_ctr_stale_page", "low_ctr_for_position"))
ACTION = {
    "low_ctr_stale_page":   "Refresh the content and rewrite title/meta",
    "low_ctr_for_position": "Rewrite title/meta description",
    "not_flagged":          "No action",
}
elig["action_label"] = elig["reason_code"].map(ACTION)

# Rank: highest score first; ties broken by impressions, then content_id, so the order is the same on every run.
queue = (elig.sort_values(["score", "impressions_90d", "content_id"], ascending=[False, False, True])
              .reset_index(drop=True))
queue["rank"] = np.arange(1, len(queue) + 1)
flagged = queue[queue["score"] > 0].copy()

OUT_COLS = ["rank", "content_id", "client_id", "score", "reason_code", "action_label",
            "impressions_90d", "avg_position", "ctr", "position_band", "band_median_ctr", "days_since_last_update"]
Path("work/outputs").mkdir(parents=True, exist_ok=True)
flagged[OUT_COLS].round({"score": 4, "band_median_ctr": 2}).to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"pages ranked: {len(queue):,} | flagged (score > 0): {len(flagged):,} | written to work/outputs/baseline_action_score.csv")
print(flagged["reason_code"].value_counts().to_string())
flagged[OUT_COLS].head(10).round(3)


pages ranked: 13,205 | flagged (score > 0): 6,487 | written to work/outputs/baseline_action_score.csv
reason_code
low_ctr_for_position    3984
low_ctr_stale_page      2503


   rank            content_id          client_id   score  \
0     1  content_c8e9d6ab9013  client_19581e27de  12.249   
1     2  content_453722754fea  client_f369cb89fc  11.257   
2     3  content_4a6607efcb46  client_6208ef0f77  11.172   
3     4  content_39881853ef0c  client_f369cb89fc  11.049   
4     5  content_8451fc6f034d  client_d029fa3a95  10.637   
5     6  content_e5f459e737b7  client_f369cb89fc  10.562   
6     7  content_d274ac4158ef  client_4e07408562  10.530   
7     8  content_0919dd345d80  client_4e07408562  10.520   
8     9  content_c84a0ab98e90  client_f369cb89fc  10.469   
9    10  content_339b357d04c7  client_bbb965ab0c  10.384   

            reason_code                                action_label  \
0    low_ctr_stale_page  Refresh the content and rewrite title/meta   
1  low_ctr_for_position              Rewrite title/meta description   
2    low_ctr_stale_page  Refresh the content and rewrite title/meta   
3  low_ctr_for_position              Rewrite title/meta

### How good is the rule?

**Precision@K** means: of the top K pages the rule flags, what fraction are actually declining. A number only means something next to the base rate (what random picking gives), so the table shows the base rate, random picking, and a "biggest pages first" ranking for comparison. I also repeat P@50 on 10 random halves of the clients.

In [6]:
# How good is the rule? Precision@K = of the top K pages flagged, how many were actually declining.
# (The outcome is used here only to grade the finished ranking.)
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(labels)[order[:k]].mean()

y = queue["declining"].values
KS = (20, 50, 100, 500)
rng = np.random.RandomState(0)
rows = {
    "my rule": [precision_at_k(queue["score"], y, k) for k in KS],
    "impressions only (biggest pages first)": [precision_at_k(queue["impressions_90d"], y, k) for k in KS],
    "random picking (average of 500 draws)": [np.mean([y[rng.permutation(len(y))[:k]].mean() for _ in range(500)]) for k in KS],
}
res = pd.DataFrame(rows, index=[f"P@{k}" for k in KS]).T.round(3)
print(f"base rate in the slice: {y.mean():.3f}")
display(res)

# Same rule, 10 different client hold-outs (the rule is fixed, so this shows how much the answer depends on WHICH clients).
clients = np.array(sorted(queue["client_id"].unique()))
rng = np.random.RandomState(1)
splits = []
for i in range(10):
    test_clients = set(rng.choice(clients, len(clients) // 2, replace=False))
    t = queue[queue["client_id"].isin(test_clients)]
    splits.append([precision_at_k(t["score"], t["declining"].values, 50), t["declining"].mean()])
splits = np.array(splits)
print(f"\nP@50 over 10 random halves of the clients: mean {splits[:,0].mean():.3f} (min {splits[:,0].min():.3f}, max {splits[:,0].max():.3f})")
print(f"base rate over the same halves:            mean {splits[:,1].mean():.3f} (min {splits[:,1].min():.3f}, max {splits[:,1].max():.3f})")
print(f"halves where the rule beat the base rate at P@50: {(splits[:,0] > splits[:,1]).sum()} of 10")

metrics = {
    "slice": {"min_impressions": MIN_IMPRESSIONS, "max_position": MAX_POSITION, "stale_days": STALE_DAYS,
              "n_pages": int(len(queue)), "n_flagged": int(len(flagged)), "base_rate": round(float(y.mean()), 4)},
    "precision_at_k_rule": {f"P@{k}": round(float(v), 4) for k, v in zip(KS, res.loc["my rule"])},
}
_ = Path("work/outputs/w04_baseline_metrics.json").write_text(json.dumps(metrics, indent=2))


base rate in the slice: 0.607


                                         P@20   P@50  P@100  P@500
my rule                                 0.650  0.640  0.720  0.706
impressions only (biggest pages first)  0.450  0.420  0.350  0.380
random picking (average of 500 draws)   0.606  0.607  0.612  0.607


P@50 over 10 random halves of the clients: mean 0.742 (min 0.640, max 0.820)
base rate over the same halves:            mean 0.630 (min 0.587, max 0.727)
halves where the rule beat the base rate at P@50: 10 of 10


**Reading the result (observed, directional).** The base rate in the slice is 60.7%. The rule gets 65% at P@20, 64% at P@50, 72% at P@100 and 70.6% at P@500, against about 61% for random picking. Ranking by impressions alone does worse than random (35-45%), so "big pages first" would be the wrong queue. Over 10 random halves of the clients the rule averages 74.2% at P@50 against a base rate of 63.0% in those halves, and beats the base rate in all 10.

**Read it with care.** The rule beats the base rate by roughly 3-4 points at P@20 and P@50, which is within the noise for such small lists, and by about 10-11 points at P@100 and P@500. I looked at the two signal tables on these same pages before fixing the rule, so these numbers are probably a little optimistic. This is the bar my Week-5 model has to clear on the same slice and the same labels.

## 3. Top-20 review

The task asks for ten reviewed pages, and a top-20 is welcome, so I review twenty. For each page: the action, the reason code, a confidence note (based on how many impressions sit behind the CTR), and what would make the pick wrong. The "what would make it wrong" lines come from the page's own numbers, for example whether its clicks are near zero or whether its client updates pages in batches.

In [7]:
# Top-20 review: one block per page - the action, the reason code, how confident I am, and what would make it wrong.
client_stats = {}
for c, g in df[df["avg_position"] > 0].groupby("client_id"):
    client_stats[c] = {"same_update_day_share": g["days_since_last_update"].value_counts(normalize=True).iloc[0]}

top20 = flagged.head(20)
for _, r in top20.iterrows():
    conf = "high" if r["impressions_90d"] >= 10000 else "medium" if r["impressions_90d"] >= 2000 else "low"
    wrong = []
    if r["ctr"] == 0 or r["ctr"] < 0.02:
        wrong.append(f"{int(r['impressions_90d']):,} impressions but almost no clicks may mean clicks are not being recorded here, not that the title is weak")
    wrong.append(f"position {r['avg_position']:.1f} is an average over many searches, so the page may rank well for one query and badly for another")
    if r["reason_code"] == "low_ctr_stale_page":
        share = client_stats[r["client_id"]]["same_update_day_share"]
        wrong.append(f"'stale' may be a bulk update date: {share:.0%} of this client's pages share one days-since-update value")
    else:
        wrong.append("the page may already have been rewritten recently, so low CTR may be old data")
    print(f"#{int(r['rank'])}  {r['action_label']}  [{r['reason_code']}]  confidence: {conf}")
    print(f"     {r['client_id']} | impressions {int(r['impressions_90d']):,} | position {r['avg_position']:.1f} | CTR {r['ctr']:.2f}% vs typical {r['band_median_ctr']:.2f}% | updated {int(r['days_since_last_update'])} days ago")
    print("     what would make it wrong: " + "; ".join(wrong) + ".")


#1  Refresh the content and rewrite title/meta  [low_ctr_stale_page]  confidence: high
     client_19581e27de | impressions 208,678 | position 9.7 | CTR 0.00% vs typical 0.20% | updated 104 days ago
     what would make it wrong: 208,678 impressions but almost no clicks may mean clicks are not being recorded here, not that the title is weak; position 9.7 is an average over many searches, so the page may rank well for one query and badly for another; 'stale' may be a bulk update date: 44% of this client's pages share one days-since-update value.
#2  Rewrite title/meta description  [low_ctr_for_position]  confidence: high
     client_f369cb89fc | impressions 140,079 | position 7.6 | CTR 0.01% vs typical 0.20% | updated 20 days ago
     what would make it wrong: 140,079 impressions but almost no clicks may mean clicks are not being recorded here, not that the title is weak; position 7.6 is an average over many searches, so the page may rank well for one query and badly for another; the pa

**What the top 20 looks like.** Seven clients appear (no client has more than 5 of the 20). Eleven of the twenty have a CTR below 0.02% against a typical 0.16-0.29%, and six are flagged as stale. Every one of them has at least 16,000 impressions, so the confidence is "high" by my impressions-based measure, which only says the numbers are big. It does not say the diagnosis is right.

The most common reason a pick could be wrong is a measurement one, not a content one: a page with tens of thousands of impressions and almost no clicks may have clicks missing in this data. The second is the "stale" tag, which for several clients is a bulk update date shared by many pages.

## 4. Weak picks + leakage check

**Weak picks.** Part A uses the outcome, only to grade, to list the top-20 pages that did not decline and to look for something they share. Part B shows who and what fills the top of the list. Part C is the leakage check: it lists the exact columns the score uses and asserts that no trend column, no future window and no outcome column is among them.

In [8]:
# Part A - which of my top 20 turned out wrong? (the outcome is used here only to grade)
t = flagged.head(20)
wrong_rows = t[t["declining"] == 0]
print(f"top 20: {int(t['declining'].sum())} declining, {len(wrong_rows)} not declining -> precision@20 = {t['declining'].mean():.2f} (base rate {y.mean():.2f})")
display(wrong_rows[["rank", "client_id", "impressions_90d", "avg_position", "ctr", "days_since_last_update", "reason_code"]].round(2))

# Do the weak picks share a trait? (small numbers - read as hints, not findings)
print("\ntop 20 by reason code:")
print(t.groupby("reason_code")["declining"].agg(n="size", declined="sum").to_string())
t100 = flagged.head(100)
near_top = t100["avg_position"] < 5
print(f"\ntop 100: position < 5 -> declining {t100[near_top]['declining'].mean():.2f} (n={int(near_top.sum())}) | position >= 5 -> declining {t100[~near_top]['declining'].mean():.2f} (n={int((~near_top).sum())})")

# Part B - what does the top of the list look like?
print("\nclients in the top 20:")
print(t["client_id"].value_counts().to_string())
top100 = flagged.head(100)
print(f"\nclients in the top 100: {top100['client_id'].nunique()} | share from the biggest client: {top100['client_id'].value_counts(normalize=True).iloc[0]:.0%}")
print(f"top 20 with zero or near-zero CTR (< 0.02%): {(t['ctr'] < 0.02).sum()}")
print(f"top 20 flagged as stale (> {STALE_DAYS} days): {int(t['stale'].sum())}")

# Part C - leakage check
USED_BY_SCORE = ["impressions_90d", "ctr", "avg_position", "days_since_last_update"]   # the ONLY columns that feed the score
FORBIDDEN = ["trend_pct", "trend_direction", "impressions_last_30d", "impressions_prev_30d",
             "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d", "declining"]
assert not set(USED_BY_SCORE) & set(FORBIDDEN), "a trend/label column leaked into the score"
product_flags = [c for c in df.columns if any(w in c.lower() for w in ("health", "quick_win", "needs_attention", "flag"))]
print(f"\ncolumns used by the score: {USED_BY_SCORE}")
print(f"forbidden columns used: none | product-flag columns present in the file: {product_flags or 'none'}")


top 20: 13 declining, 7 not declining -> precision@20 = 0.65 (base rate 0.61)


    rank          client_id  impressions_90d  avg_position   ctr  \
2      3  client_6208ef0f77           128068           2.2  0.01   
4      5  client_d029fa3a95           272144           2.3  0.03   
6      7  client_4e07408562            65138           6.8  0.01   
8      9  client_f369cb89fc           223271           7.8  0.03   
9     10  client_bbb965ab0c            46879           3.7  0.01   
15    16  client_19581e27de           123469           8.0  0.03   
19    20  client_6208ef0f77            33286           4.6  0.02   

    days_since_last_update           reason_code  
2                      104    low_ctr_stale_page  
4                       20  low_ctr_for_position  
6                       26  low_ctr_for_position  
8                       20  low_ctr_for_position  
9                       15  low_ctr_for_position  
15                     104    low_ctr_stale_page  
19                     104    low_ctr_stale_page  


top 20 by reason code:
                       n  declined
reason_code                       
low_ctr_for_position  14        10
low_ctr_stale_page     6         3

top 100: position < 5 -> declining 0.62 (n=16) | position >= 5 -> declining 0.74 (n=84)

clients in the top 20:
client_id
client_19581e27de    5
client_4e07408562    5
client_f369cb89fc    4
client_6208ef0f77    2
client_7f2253d7e2    2
client_d029fa3a95    1
client_bbb965ab0c    1

clients in the top 100: 14 | share from the biggest client: 44%
top 20 with zero or near-zero CTR (< 0.02%): 11
top 20 flagged as stale (> 90 days): 6

columns used by the score: ['impressions_90d', 'ctr', 'avg_position', 'days_since_last_update']
forbidden columns used: none | product-flag columns present in the file: none


**Weak picks (small numbers, read as hints).** 13 of the top 20 were declining and 7 were not (ranks 3, 5, 7, 9, 10, 16 and 20). Four of the seven weak picks sit at position 2-5 (ranks 3, 5, 10 and 20), and none of the four was declining. Among the top 100, pages at position below 5 decline 62% of the time (n = 16) and the rest 74% (n = 84). That is too few pages to conclude anything, and across the whole slice the 1-3 band does show a strong low-CTR effect, but a page ranking 2nd with a CTR of 0.01% is extreme enough that I would check the data before rewriting anything. The six "stale" picks were declining 3 times out of 6, against 10 out of 14 for the other picks, which fits the MIXED verdict on staleness.

**Concentration.** The top 100 comes from 14 clients and the largest one supplies 44% of it, so the queue is not spread evenly. A real queue should probably be ranked within each client as well.

**Leakage check.** The score uses only `impressions_90d`, `ctr`, `avg_position` and `days_since_last_update`. No trend column (`trend_pct`, `trend_direction`), no last-30 or previous-30 column, and no outcome column is used, and the file has no product-flag columns. One honest limit remains: these four columns are 90-day totals measured over the same 90 days in which the 30-day trend behind the outcome was computed. Nothing comes from after the outcome, but this is a snapshot, not a clean "past predicts future" split. That has to be rebuilt with a real time split before anyone treats the numbers as a forecast.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.